# 🛒 Module 5: Market Basket Analysis (FP-Growth / Apriori)

In this notebook, we construct a Transaction Item Matrix from cleaned retail orders, execute the **FP-Growth Algorithm** to find frequent itemsets, generate **Association Rules**, and evaluate metrics (**Support, Confidence, Lift**).

---

In [1]:
import os
import pandas as pd
import numpy as np
from mlxtend.frequent_patterns import fpgrowth, association_rules
import plotly.express as px

cleaned_path = os.path.join("..", "data", "processed", "cleaned_retail_data.csv")
df = pd.read_csv(cleaned_path)
print(f"Loaded Cleaned Data: {df.shape[0]:,} rows")

Loaded Cleaned Data: 392,692 rows


### Step 1: Construct One-Hot Encoded Basket Matrix

In [2]:
top_items = df['Description'].value_counts().head(1200).index
df_filtered = df[df['Description'].isin(top_items)]

basket = (df_filtered.groupby(['InvoiceNo', 'Description'])['Quantity']
          .sum().unstack().reset_index().fillna(0)
          .set_index('InvoiceNo'))

basket_sets = (basket > 0)
print(f"Transaction Matrix Shape: {basket_sets.shape}")

Transaction Matrix Shape: (18154, 1200)


### Step 2: FP-Growth Frequent Itemsets

In [3]:
frequent_itemsets = fpgrowth(basket_sets, min_support=0.01, use_colnames=True)
print(f"Found {len(frequent_itemsets)} frequent itemsets.")
frequent_itemsets.head(10)

Found 1017 frequent itemsets.


,support,itemsets
0,0.108571,frozenset({WHITE HANGING HEART T-LIGHT HOLDER})
1,0.017682,frozenset({RED WOOLLY HOTTIE WHITE HEART.})
2,0.017572,frozenset({KNITTED UNION FLAG HOT WATER BOTTLE})
3,0.016360,frozenset({SET 7 BABUSHKA NESTING BOXES})
4,0.013275,frozenset({CREAM CUPID HEARTS COAT HANGER})
5,0.013110,frozenset({WHITE METAL LANTERN})
6,0.024292,frozenset({HAND WARMER UNION JACK})
7,0.075741,frozenset({ASSORTED COLOUR BIRD ORNAMENT})
8,0.038063,frozenset({HOME BUILDING BLOCK WORD})
9,0.030572,frozenset({LOVE BUILDING BLOCK WORD})


### Step 3: Generate Association Rules (Support, Confidence, Lift)

In [4]:
rules = association_rules(frequent_itemsets, metric="lift", min_threshold=0.2)
rules = rules.sort_values('lift', ascending=False).reset_index(drop=True)
print(f"Generated {len(rules)} Association Rules!")

# Format frozenset to strings for clean viewing
rules['antecedents_str'] = rules['antecedents'].apply(lambda x: ', '.join(list(x)))
rules['consequents_str'] = rules['consequents'].apply(lambda x: ', '.join(list(x)))

rules[['antecedents_str', 'consequents_str', 'support', 'confidence', 'lift']].head(20)

Generated 1030 Association Rules!


,antecedents_str,consequents_str,support,confidence,lift
0,REGENCY TEA PLATE PINK,"REGENCY TEA PLATE ROSES , REGENCY TEA PLATE GR...",0.010191,0.825893,65.472746
1,"REGENCY TEA PLATE ROSES , REGENCY TEA PLATE GR...",REGENCY TEA PLATE PINK,0.010191,0.807860,65.472746
2,"REGENCY TEA PLATE ROSES , REGENCY TEA PLATE PINK",REGENCY TEA PLATE GREEN,0.010191,0.939086,63.141380
3,REGENCY TEA PLATE GREEN,"REGENCY TEA PLATE ROSES , REGENCY TEA PLATE PINK",0.010191,0.685185,63.141380
4,REGENCY TEA PLATE GREEN,REGENCY TEA PLATE PINK,0.011127,0.748148,60.633399
5,REGENCY TEA PLATE PINK,REGENCY TEA PLATE GREEN,0.011127,0.901786,60.633399
6,POPPY'S PLAYHOUSE LIVINGROOM,"POPPY'S PLAYHOUSE KITCHEN, POPPY'S PLAYHOUSE B...",0.010246,0.738095,52.753468
7,"POPPY'S PLAYHOUSE KITCHEN, POPPY'S PLAYHOUSE B...",POPPY'S PLAYHOUSE LIVINGROOM,0.010246,0.732283,52.753468
8,REGENCY SUGAR BOWL GREEN,REGENCY MILK JUG PINK,0.011347,0.768657,51.302184
9,REGENCY MILK JUG PINK,REGENCY SUGAR BOWL GREEN,0.011347,0.757353,51.302184


### Step 4: Export Rules to CSV

In [5]:
rules_to_save = rules.copy()
rules_to_save['antecedents'] = rules_to_save['antecedents_str']
rules_to_save['consequents'] = rules_to_save['consequents_str']
rules_to_save.drop(columns=['antecedents_str', 'consequents_str'], inplace=True)

out_rules = os.path.join("..", "data", "processed", "association_rules.csv")
rules_to_save.to_csv(out_rules, index=False)
print(f"Association rules exported to: {out_rules}")

Association rules exported to: ..\data\processed\association_rules.csv
